In [ ]:
!pip install Unsloth
!pip install google.colab
!pip install huggingface_hub
!pip install trl
!pip install transformers
!pip install os

In [ ]:
import torch
# 1. Install Unsloth & Dependencies
!pip install --no-deps "xformers<0.0.27" "trl<0.9.0" peft accelerate bitsandbytes
!pip install "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"

# 2. Login to Hugging Face
from google.colab import userdata
from huggingface_hub import login

try:
    hf_token = userdata.get('HF_TOKEN')
    login(hf_token)
    print("Successfully logged into Hugging Face!")
except:
    print("Error: HF_TOKEN not found in Colab Secrets. Check the secrets tab.")

In [ ]:
from datasets import load_dataset

# Load the file
dataset = load_dataset("json", data_files="dataset.jsonl", split="train")

print(f"Dataset Loaded! Total examples: {len(dataset)}")
print("Sample Format Check:", dataset[0]["text"][:150], "...")

In [ ]:
from unsloth import FastLanguageModel
import torch

max_seq_length = 2048
dtype = None
load_in_4bit = True

# Load Model
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/gemma-3-1b-it-bnb-4bit", # Gemma 3 1B Instruct
    max_seq_length = max_seq_length,
    dtype = dtype,
    load_in_4bit = load_in_4bit,
)

# Apply LoRA (PEFT)
model = FastLanguageModel.get_peft_model(
    model,
    r = 16,
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj",
                      "gate_proj", "up_proj", "down_proj",],
    lora_alpha = 16,
    lora_dropout = 0,
    bias = "none",
    use_gradient_checkpointing = "unsloth",
)
print("Gemma 3:1B is ready for training.")

In [ ]:
from trl import SFTTrainer
from transformers import TrainingArguments

# 1. Configure Trainer
trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = dataset,
    dataset_text_field = "text",
    max_seq_length = max_seq_length,
    args = TrainingArguments(
        per_device_train_batch_size = 4, # Optimized for 1B model
        gradient_accumulation_steps = 4,
        warmup_steps = 5,
        max_steps = 60,
        learning_rate = 2e-4,
        fp16 = not torch.cuda.is_bf16_supported(),
        bf16 = torch.cuda.is_bf16_supported(),
        logging_steps = 1,
        output_dir = "outputs",
        optim = "adamw_8bit",
    ),
)

# 2. Run Training
trainer.train()

# 3. Quick Vibe Check (Testing)
FastLanguageModel.for_inference(model)
test_prompt = "<start_of_turn>user\nAnalyze this URL: http://signin-apple-id.secure-login.xyz/<end_of_turn>\n<start_of_turn>model\n"
inputs = tokenizer([test_prompt], return_tensors = "pt").to("cuda")
outputs = model.generate(**inputs, max_new_tokens = 64)

print("\n--- TEST RESULT ---")
print(tokenizer.batch_decode(outputs)[0])

In [ ]:
import os
from google.colab import drive

# 1. Ensure Drive is actually mounted
if not os.path.exists("/content/drive/MyDrive"):
    drive.mount('/content/drive')

# 2. Define your folder name in Drive
save_directory = "/content/drive/MyDrive/Gemma3_Phishing_Project"

# 3. Save the LoRA adapters (the small tuning files)
model.save_pretrained_merged(
    os.path.join(save_directory, "lora_model"),
    tokenizer,
    save_method = "lora"
)

# 4. Save as GGUF (The single file for Ollama/Local use)
# This handles the conversion and the move in one step
model.save_pretrained_gguf(
    os.path.join(save_directory, "gemma3_1b_phishing"),
    tokenizer,
    quantization_method = "q4_k_m"
)

print(f"✅ Success! Your files are now in Google Drive folder: {save_directory}")